# Flight Price SQL + MLlib Starter
Notebook inicial para validar o ambiente PySpark, executar consultas SQL e treinar modelos basicos com pyspark.ml.
Diretrizes deste notebook:
- leitura priorizando Parquet no HDFS, com fallback para CSV apenas quando necessario;
- preparacao, validacao e exploracao usando spark.sql(...);
- modelagem com MLlib sobre uma base preparada integralmente em SQL;
- comparacao entre cenarios com e sem ase_fare;
- split temporal para reduzir vazamento entre treino e teste.


In [ ]:
import os
from IPython.display import display
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import OneHotEncoder, StringIndexer, VectorAssembler
from pyspark.ml.regression import DecisionTreeRegressor, LinearRegression


In [ ]:
APP_NAME = "flight-price-sql-mllib"
SPARK_MASTER_URL = os.environ.get("SPARK_MASTER", "spark://spark-master:7077")

# Parquet is the canonical dataset (~6.8 GB, snappy-compressed, partitioned by
# startingAirport). Run `make convert-local` from the project root if HDFS does
# not yet have it.
HDFS_PARQUET_PATH  = "hdfs://namenode:9000/data/itineraries.parquet"
LOCAL_PARQUET_PATH = "/data/itineraries.parquet"

# Default load sample. Drop to e.g. 0.1 to iterate faster on the ML pipeline.
LOAD_SAMPLE_RATIO = 0.1
DEV_SAMPLE_RATIO = 1.0


def get_or_create_spark(app_name: str = APP_NAME) -> SparkSession:
    spark_session = (
        SparkSession.builder
        .appName(app_name)
        .master(SPARK_MASTER_URL)
        .config("spark.sql.shuffle.partitions", "16")
        .config("spark.sql.session.timeZone", "UTC")
        .config("spark.sql.legacy.timeParserPolicy", "LEGACY")
        .config("spark.sql.repl.eagerEval.enabled", "true")
        .config("spark.sql.repl.eagerEval.maxNumRows", "20")
        .config("spark.sql.repl.eagerEval.truncate", "80")
        .getOrCreate()
    )
    spark_session.sparkContext.setLogLevel("WARN")
    return spark_session


def resolve_parquet_paths(spark_session: SparkSession) -> list[str]:
    """Return Parquet paths to try, in preference order."""
    if spark_session.sparkContext.master.startswith("local"):
        return [HDFS_PARQUET_PATH, LOCAL_PARQUET_PATH]
    return [HDFS_PARQUET_PATH]


def load_flights(spark_session: SparkSession, candidate_paths: list[str]):
    """Read the Parquet dataset; raise if no candidate is reachable."""
    last_error = None
    print(f"Tentando carregar Parquet pelos caminhos: {candidate_paths}")
    for path in candidate_paths:
        try:
            df = spark_session.read.parquet(path)
            print(f"Dataset registrado: {path}")
            return df, path
        except Exception as exc:
            print(f"Falha ao carregar {path}: {exc}")
            last_error = exc
    raise RuntimeError(
        "Nao foi possivel carregar o dataset Parquet. "
        "Rode `make convert-local` no projeto para gera-lo."
    ) from last_error


def apply_load_sample(df, ratio: float):
    if ratio >= 1.0:
        print("Usando dataset completo apos a leitura.")
        return df
    print(f"Aplicando amostra de carga com ratio={ratio} para acelerar as iteracoes.")
    return df.sample(withReplacement=False, fraction=ratio, seed=42)


def run_sql(query: str, preview_rows: int = 20):
    df = spark.sql(query)
    display(df.limit(preview_rows).toPandas())
    return df


In [ ]:
spark = get_or_create_spark()
candidate_paths = resolve_parquet_paths(spark)
raw_input_df, source_path = load_flights(spark, candidate_paths)
raw_df = apply_load_sample(raw_input_df, LOAD_SAMPLE_RATIO)
raw_df.createOrReplaceTempView("flights_raw")

print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)
print("Source path:", source_path)
print("Load sample ratio:", LOAD_SAMPLE_RATIO)


In [ ]:
run_sql("SELECT * FROM flights_raw LIMIT 5")


In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW flights_clean AS
WITH base AS (
    SELECT
        legId AS leg_id,
        TO_DATE(searchDate) AS search_date,
        TO_DATE(flightDate) AS flight_date,
        startingAirport AS starting_airport,
        destinationAirport AS destination_airport,
        fareBasisCode AS fare_basis_code,
        UPPER(SUBSTRING(COALESCE(fareBasisCode, 'UNK'), 1, 1)) AS fare_basis_prefix,
        travelDuration AS travel_duration_iso,
        CAST(elapsedDays AS INT) AS overnight_days,
        DATEDIFF(TO_DATE(flightDate), TO_DATE(searchDate)) AS days_until_flight,
        CASE WHEN LOWER(CAST(isBasicEconomy AS STRING)) = 'true' THEN 1 ELSE 0 END AS is_basic_economy,
        CASE WHEN LOWER(CAST(isRefundable AS STRING)) = 'true' THEN 1 ELSE 0 END AS is_refundable,
        CASE WHEN LOWER(CAST(isNonStop AS STRING)) = 'true' THEN 1 ELSE 0 END AS is_non_stop,
        CAST(baseFare AS DOUBLE) AS base_fare,
        CAST(totalFare AS DOUBLE) AS total_fare,
        CAST(seatsRemaining AS INT) AS seats_remaining,
        CAST(totalTravelDistance AS DOUBLE) AS total_travel_distance,
        (
            CAST(CASE WHEN REGEXP_EXTRACT(travelDuration, '([0-9]+)D', 1) = '' THEN '0' ELSE REGEXP_EXTRACT(travelDuration, '([0-9]+)D', 1) END AS INT) * 1440
            + CAST(CASE WHEN REGEXP_EXTRACT(travelDuration, '([0-9]+)H', 1) = '' THEN '0' ELSE REGEXP_EXTRACT(travelDuration, '([0-9]+)H', 1) END AS INT) * 60
            + CAST(CASE WHEN REGEXP_EXTRACT(travelDuration, '([0-9]+)M', 1) = '' THEN '0' ELSE REGEXP_EXTRACT(travelDuration, '([0-9]+)M', 1) END AS INT)
        ) AS travel_duration_minutes,
        CAST(NULLIF(REGEXP_EXTRACT(COALESCE(segmentsDepartureTimeRaw, ''), 'T([0-9]{2}):', 1), '') AS INT) AS departure_hour,
        COALESCE(segmentsDepartureAirportCode, '') AS segments_departure_airport_code_raw,
        FILTER(SPLIT(REPLACE(COALESCE(segmentsDepartureAirportCode, ''), '||', '~'), '~'), x -> TRIM(x) <> '') AS departure_airport_segments,
        FILTER(SPLIT(REPLACE(COALESCE(segmentsDurationInSeconds, ''), '||', '~'), '~'), x -> TRIM(x) <> '') AS duration_seconds_segments,
        FILTER(SPLIT(REPLACE(COALESCE(segmentsDistance, ''), '||', '~'), '~'), x -> TRIM(x) <> '' AND LOWER(TRIM(x)) <> 'none') AS distance_segments,
        FILTER(SPLIT(REPLACE(COALESCE(segmentsAirlineCode, ''), '||', '~'), '~'), x -> TRIM(x) <> '') AS airline_segments,
        FILTER(SPLIT(REPLACE(COALESCE(segmentsCabinCode, ''), '||', '~'), '~'), x -> TRIM(x) <> '') AS cabin_segments,
        DAYOFWEEK(TO_DATE(flightDate)) AS flight_day_of_week,
        MONTH(TO_DATE(flightDate)) AS flight_month,
        CONCAT(startingAirport, '-', destinationAirport) AS route
    FROM flights_raw
    WHERE totalFare IS NOT NULL
), prepared AS (
    SELECT
        *,
        CASE
            WHEN segments_departure_airport_code_raw IS NULL OR TRIM(segments_departure_airport_code_raw) = '' THEN 0
            ELSE CAST(((LENGTH(segments_departure_airport_code_raw) - LENGTH(REPLACE(segments_departure_airport_code_raw, '||', ''))) / 2) + 1 AS INT)
        END AS segment_count,
        AGGREGATE(duration_seconds_segments, CAST(0.0 AS DOUBLE), (acc, x) -> acc + COALESCE(CAST(x AS DOUBLE), 0.0D)) / 60.0 AS segment_duration_sum_minutes,
        AGGREGATE(distance_segments, CAST(0.0 AS DOUBLE), (acc, x) -> acc + COALESCE(CAST(x AS DOUBLE), 0.0D)) AS known_segment_distance,
        SIZE(distance_segments) AS known_segment_distance_count,
        SIZE(ARRAY_DISTINCT(airline_segments)) AS distinct_airline_count,
        SIZE(ARRAY_DISTINCT(cabin_segments)) AS distinct_cabin_count
    FROM base
), engineered AS (
    SELECT
        *,
        GREATEST(segment_count - 1, 0) AS stop_count,
        CASE WHEN segment_count > known_segment_distance_count THEN 1 ELSE 0 END AS has_missing_segment_distance,
        CASE WHEN segment_count > 0 THEN known_segment_distance / segment_count END AS distance_per_segment,
        CASE WHEN segment_count > 0 THEN segment_duration_sum_minutes / segment_count END AS segment_duration_avg_minutes,
        CASE
            WHEN total_travel_distance IS NULL OR isnan(total_travel_distance) THEN known_segment_distance
            ELSE total_travel_distance
        END AS effective_distance,
        GREATEST(travel_duration_minutes - segment_duration_sum_minutes, 0.0D) AS travel_minus_segment_minutes,
        CASE
            WHEN segment_count <= 1 THEN 0.0D
            ELSE GREATEST(travel_duration_minutes - segment_duration_sum_minutes, 0.0D)
        END AS layover_minutes,
        CASE WHEN flight_day_of_week IN (1, 7) THEN 1 ELSE 0 END AS is_weekend
    FROM prepared
)
SELECT
    leg_id,
    search_date,
    flight_date,
    starting_airport,
    destination_airport,
    fare_basis_code,
    fare_basis_prefix,
    travel_duration_iso,
    overnight_days,
    days_until_flight,
    is_basic_economy,
    is_refundable,
    is_non_stop,
    base_fare,
    total_fare,
    seats_remaining,
    total_travel_distance,
    travel_duration_minutes,
    departure_hour,
    segment_count,
    stop_count,
    segment_duration_sum_minutes,
    segment_duration_avg_minutes,
    known_segment_distance,
    known_segment_distance_count,
    effective_distance,
    distance_per_segment,
    layover_minutes,
    travel_minus_segment_minutes,
    has_missing_segment_distance,
    distinct_airline_count,
    distinct_cabin_count,
    flight_day_of_week,
    flight_month,
    is_weekend,
    route
FROM engineered
""")
if DEV_SAMPLE_RATIO < 1.0:
    spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW flights_dev AS
    SELECT *
    FROM flights_clean
    WHERE rand(42) <= {DEV_SAMPLE_RATIO}
    """)
else:
    spark.sql("CREATE OR REPLACE TEMP VIEW flights_dev AS SELECT * FROM flights_clean")


In [ ]:
run_sql("""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN days_until_flight < 0 THEN 1 ELSE 0 END) AS invalid_days_until_flight,
    SUM(CASE WHEN segment_count <= 0 THEN 1 ELSE 0 END) AS invalid_segment_count,
    SUM(CASE WHEN segment_count > 4 THEN 1 ELSE 0 END) AS invalid_segment_upper_bound,
    SUM(CASE WHEN travel_duration_minutes <= 0 THEN 1 ELSE 0 END) AS invalid_travel_duration,
    SUM(CASE WHEN departure_hour IS NOT NULL AND (departure_hour < 0 OR departure_hour > 23) THEN 1 ELSE 0 END) AS invalid_departure_hour,
    SUM(CASE WHEN layover_minutes < 0 THEN 1 ELSE 0 END) AS negative_layover_signal,
    SUM(CASE WHEN is_non_stop = 1 AND stop_count <> 0 THEN 1 ELSE 0 END) AS invalid_non_stop_stop_count,
    SUM(CASE WHEN is_non_stop = 1 AND layover_minutes > 15 THEN 1 ELSE 0 END) AS non_stop_with_layover,
    MIN(total_fare) AS min_total_fare,
    MAX(total_fare) AS max_total_fare,
    ROUND(AVG(total_fare), 2) AS avg_total_fare,
    ROUND(AVG(travel_duration_minutes), 2) AS avg_duration_minutes,
    ROUND(AVG(days_until_flight), 2) AS avg_days_until_flight,
    ROUND(AVG(layover_minutes), 2) AS avg_layover_minutes
FROM flights_dev
""")


In [ ]:
run_sql("""
SELECT
    segment_count,
    stop_count,
    COUNT(*) AS total_voos,
    ROUND(AVG(total_fare), 2) AS avg_total_fare,
    ROUND(AVG(travel_duration_minutes), 2) AS avg_duration_minutes,
    ROUND(AVG(layover_minutes), 2) AS avg_layover_minutes,
    ROUND(AVG(distinct_airline_count), 2) AS avg_distinct_airlines
FROM flights_dev
GROUP BY segment_count, stop_count
ORDER BY segment_count, stop_count
LIMIT 20
""")


In [ ]:
run_sql("""
SELECT
    route,
    travel_duration_iso,
    travel_duration_minutes,
    segment_count,
    stop_count,
    departure_hour,
    layover_minutes,
    days_until_flight,
    total_fare
FROM flights_dev
WHERE travel_duration_minutes <= 0
   OR segment_count <= 0
   OR days_until_flight < 0
   OR (departure_hour IS NOT NULL AND (departure_hour < 0 OR departure_hour > 23))
ORDER BY total_fare DESC
LIMIT 20
""")


In [ ]:
run_sql("""
SELECT
    route,
    COUNT(*) AS total_voos,
    ROUND(AVG(total_fare), 2) AS avg_total_fare,
    ROUND(AVG(base_fare), 2) AS avg_base_fare
FROM flights_dev
GROUP BY route
ORDER BY avg_total_fare DESC
LIMIT 20
""")


In [ ]:
run_sql("""
SELECT
    is_non_stop,
    stop_count,
    COUNT(*) AS total_voos,
    ROUND(AVG(total_fare), 2) AS avg_total_fare,
    ROUND(AVG(days_until_flight), 2) AS avg_days_until_flight,
    ROUND(AVG(layover_minutes), 2) AS avg_layover_minutes
FROM flights_dev
GROUP BY is_non_stop, stop_count
ORDER BY is_non_stop DESC, stop_count ASC
""")


In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW flights_ml_base AS
SELECT
    total_fare,
    base_fare,
    days_until_flight,
    overnight_days,
    seats_remaining,
    COALESCE(effective_distance, 0.0D) AS total_travel_distance,
    travel_duration_minutes,
    segment_count,
    stop_count,
    flight_day_of_week,
    flight_month,
    is_basic_economy,
    is_refundable,
    is_non_stop,
    route,
    flight_date,
    starting_airport,
    destination_airport,
    fare_basis_prefix,
    departure_hour,
    segment_duration_sum_minutes,
    segment_duration_avg_minutes,
    effective_distance,
    distance_per_segment,
    layover_minutes,
    has_missing_segment_distance,
    distinct_airline_count,
    distinct_cabin_count,
    is_weekend,
    travel_minus_segment_minutes
FROM flights_dev
WHERE total_fare IS NOT NULL
  AND base_fare IS NOT NULL
  AND days_until_flight IS NOT NULL
  AND overnight_days IS NOT NULL
  AND seats_remaining IS NOT NULL
  AND travel_duration_minutes IS NOT NULL
  AND departure_hour IS NOT NULL
  AND segment_count BETWEEN 1 AND 4
  AND departure_hour BETWEEN 0 AND 23
  AND travel_duration_minutes BETWEEN 30 AND 4320
  AND segment_duration_sum_minutes > 0
  AND days_until_flight BETWEEN 0 AND 365
  AND layover_minutes BETWEEN 0 AND 1440
  AND (is_non_stop = 0 OR stop_count = 0)
  AND (is_non_stop = 0 OR layover_minutes <= 15)
""")
run_sql("""
SELECT
    COUNT(*) AS total_rows,
    MIN(flight_date) AS min_flight_date,
    MAX(flight_date) AS max_flight_date,
    ROUND(AVG(total_fare), 2) AS avg_total_fare,
    ROUND(AVG(days_until_flight), 2) AS avg_days_until_flight,
    ROUND(AVG(segment_count), 2) AS avg_segment_count,
    ROUND(AVG(layover_minutes), 2) AS avg_layover_minutes
FROM flights_ml_base
""")
cutoff_unix = spark.sql("SELECT percentile_approx(unix_timestamp(flight_date), 0.8) AS cutoff_unix FROM flights_ml_base").first()["cutoff_unix"]
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW flights_train AS
SELECT *
FROM flights_ml_base
WHERE unix_timestamp(flight_date) <= {cutoff_unix}
""")
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW flights_test AS
SELECT *
FROM flights_ml_base
WHERE unix_timestamp(flight_date) > {cutoff_unix}
""")
train_df = spark.sql("SELECT * FROM flights_train")
test_df = spark.sql("SELECT * FROM flights_test")
train_df.cache()
test_df.cache()
run_sql("""
SELECT 'train' AS split_name, COUNT(*) AS total_rows, MIN(flight_date) AS min_flight_date, MAX(flight_date) AS max_flight_date FROM flights_train
UNION ALL
SELECT 'test' AS split_name, COUNT(*) AS total_rows, MIN(flight_date) AS min_flight_date, MAX(flight_date) AS max_flight_date FROM flights_test
""")
print("Train rows:", train_df.count())
print("Test rows:", test_df.count())


In [ ]:
COMMON_FEATURE_COLS = [
    "days_until_flight",
    "overnight_days",
    "seats_remaining",
    "total_travel_distance",
    "travel_duration_minutes",
    "segment_count",
    "stop_count",
    "flight_day_of_week",
    "flight_month",
    "is_basic_economy",
    "is_refundable",
    "is_non_stop",
    "departure_hour",
    "segment_duration_sum_minutes",
    "segment_duration_avg_minutes",
    "effective_distance",
    "distance_per_segment",
    "layover_minutes",
    "has_missing_segment_distance",
    "distinct_airline_count",
    "distinct_cabin_count",
    "is_weekend",
    "travel_minus_segment_minutes"
]
CATEGORICAL_FEATURE_COLS = [
    "route",
    "starting_airport",
    "destination_airport",
    "fare_basis_prefix"
]
FEATURE_SETS = {
    "with_base_fare": ["base_fare"] + COMMON_FEATURE_COLS,
    "without_base_fare": COMMON_FEATURE_COLS
}
regression_evaluators = {
    "rmse": RegressionEvaluator(labelCol="total_fare", predictionCol="prediction", metricName="rmse"),
    "mae": RegressionEvaluator(labelCol="total_fare", predictionCol="prediction", metricName="mae"),
    "r2": RegressionEvaluator(labelCol="total_fare", predictionCol="prediction", metricName="r2")
}
train_count = train_df.count()
test_count = test_df.count()
def build_pipeline(feature_cols, estimator):
    indexers = []
    index_output_cols = []
    encoded_output_cols = []
    for column_name in CATEGORICAL_FEATURE_COLS:
        index_col = f"{column_name}_index"
        encoded_col = f"{column_name}_ohe"
        indexers.append(StringIndexer(inputCol=column_name, outputCol=index_col, handleInvalid="keep"))
        index_output_cols.append(index_col)
        encoded_output_cols.append(encoded_col)
    encoder = OneHotEncoder(inputCols=index_output_cols, outputCols=encoded_output_cols)
    assembler = VectorAssembler(inputCols=feature_cols + encoded_output_cols, outputCol="features")
    return Pipeline(stages=indexers + [encoder, assembler, estimator])
def train_and_evaluate(model_name: str, estimator, feature_set_name: str, feature_cols):
    pipeline = build_pipeline(feature_cols, estimator)
    fitted_pipeline = pipeline.fit(train_df)
    predictions = fitted_pipeline.transform(test_df)
    metrics = {
        "model": model_name,
        "feature_set": feature_set_name,
        "uses_base_fare": 1 if "base_fare" in feature_cols else 0,
        "rmse": regression_evaluators["rmse"].evaluate(predictions),
        "mae": regression_evaluators["mae"].evaluate(predictions),
        "r2": regression_evaluators["r2"].evaluate(predictions),
        "train_rows": train_count,
        "test_rows": test_count
    }
    return fitted_pipeline, predictions, metrics


In [ ]:
model_runs = {}
results = []
for feature_set_name, feature_cols in FEATURE_SETS.items():
    linear_regression = LinearRegression(
        featuresCol="features",
        labelCol="total_fare",
        predictionCol="prediction",
        maxIter=20,
        regParam=0.1,
        elasticNetParam=0.0
    )
    decision_tree = DecisionTreeRegressor(
        featuresCol="features",
        labelCol="total_fare",
        predictionCol="prediction",
        maxDepth=10,
        minInstancesPerNode=100
    )
    for base_model_name, estimator in [
        ("linear_regression", linear_regression),
        ("decision_tree", decision_tree)
    ]:
        run_name = f"{base_model_name}__{feature_set_name}"
        fitted_pipeline, predictions, metrics = train_and_evaluate(run_name, estimator, feature_set_name, feature_cols)
        model_runs[run_name] = {
            "pipeline": fitted_pipeline,
            "predictions": predictions,
            "metrics": metrics
        }
        results.append(metrics)
metrics_df = spark.createDataFrame(results)
display(metrics_df.orderBy("rmse").toPandas())


In [ ]:
best_run_name = metrics_df.orderBy("rmse").first()["model"]
best_no_base_run_name = metrics_df.filter("uses_base_fare = 0").orderBy("rmse").first()["model"]
best_predictions = model_runs[best_run_name]["predictions"]
best_predictions.createOrReplaceTempView("best_predictions")
best_no_base_predictions = model_runs[best_no_base_run_name]["predictions"]
best_no_base_predictions.createOrReplaceTempView("best_no_base_predictions")
print("Best overall run:", best_run_name)
print("Best no-base-fare run:", best_no_base_run_name)
run_sql("""
SELECT
    stop_count,
    COUNT(*) AS total_voos,
    ROUND(AVG(ABS(total_fare - prediction)), 2) AS avg_absolute_error,
    ROUND(MAX(ABS(total_fare - prediction)), 2) AS max_absolute_error,
    ROUND(AVG(total_fare), 2) AS avg_total_fare
FROM best_no_base_predictions
GROUP BY stop_count
ORDER BY stop_count
""")
run_sql("""
SELECT
    route,
    total_fare,
    ROUND(prediction, 2) AS prediction,
    ROUND(ABS(total_fare - prediction), 2) AS absolute_error,
    days_until_flight,
    travel_duration_minutes,
    layover_minutes,
    segment_count,
    stop_count,
    is_non_stop,
    distinct_airline_count,
    fare_basis_prefix
FROM best_no_base_predictions
ORDER BY absolute_error DESC
LIMIT 20
""")


## Proximos passos
- gerar hdfs://namenode:9000/data/itineraries.parquet com make convert-local e priorizar esse caminho para os proximos reruns;
- subir LOAD_SAMPLE_RATIO para 1.0 quando quiser benchmark mais forte e o cluster estiver estavel;
- manter a leitura with_base_fare como baseline de teto e usar without_base_fare como avaliacao mais realista;
- testar RandomForestRegressor e GBTRegressor como proximos baselines quando o pipeline estiver consolidado.


In [ ]:
print("Notebook pronto para exploracao interativa no Jupyter Lab.")
